# DL-02 — Train U-Net and DeepLabV3+ for Geospatial Suitability Segmentation

This notebook trains deep learning segmentation models using the patch dataset generated from **DL-01 — Prepare Raster Labels and Patches**.

**Research context**: Multi-source geospatial deep learning framework for mapping agro-industrial residue potential as biomass-derived carbon nanomaterial feedstock in West Java, Indonesia.

## Main tasks
1. Load patch dataset from DL-01.
2. Audit patch arrays and class distribution.
3. Split patches into train, validation, and test sets.
4. Train U-Net baseline.
5. Train DeepLabV3+ model.
6. Evaluate Accuracy, Macro-F1, Mean IoU, and class-level IoU.
7. Save model weights, training history, confusion matrices, and prediction rasters.

## Important note
The labels used here are **proxy labels** derived from the rule-based suitability index and ML-ready geospatial dataset. Therefore, model results should be interpreted as deep learning approximation of spatial suitability labels, not direct field-validated residue observations.


## Step 0 — Runtime setup

Recommended Colab runtime:

- Runtime type: **GPU**
- GPU: T4 or better
- Python: default Colab Python

Expected input folder:

```text
/content/drive/MyDrive/geospatial_biomass_carbon/deep_learning/DL_01_prepare_raster_labels_and_patches/
```

Expected main input file:

```text
patch_dataset_classification_regression.npz
```


In [ ]:
# ============================================================
# Step 1 — Install dependencies
# ============================================================

!pip install -q rasterio geopandas pyogrio openpyxl scikit-learn matplotlib tqdm segmentation-models-pytorch


In [ ]:
# ============================================================
# Step 2 — Imports and Google Drive mount
# ============================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import rasterio
from rasterio.transform import Affine

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, precision_score, recall_score

from google.colab import drive

drive.mount('/content/drive')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# Step 3 — Paths and training configuration
# ============================================================

BASE_DIR = Path('/content/drive/MyDrive/geospatial_biomass_carbon')
DL01_DIR = BASE_DIR / 'deep_learning' / 'DL_01_prepare_raster_labels_and_patches'
DL02_DIR = BASE_DIR / 'deep_learning' / 'DL_02_train_unet_deeplabv3_geospatial'
DL02_DIR.mkdir(parents=True, exist_ok=True)

NPZ_FILE = DL01_DIR / 'patch_dataset_classification_regression.npz'
PATCH_METADATA_FILE = DL01_DIR / 'patch_metadata.csv'
LABEL_CLASS_TIF = DL01_DIR / 'label_potential_class_final.tif'

# Training settings
PATCH_SIZE = 128
BATCH_SIZE = 8          # Lower to 4 if GPU memory is limited
EPOCHS_UNET = 25
EPOCHS_DEEPLAB = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# Label settings from DL-01
CLASS_NODATA = 255
CLASS_NAMES = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
NUM_CLASSES = len(CLASS_NAMES)

# Model selection
TRAIN_UNET = True
TRAIN_DEEPLABV3PLUS = True

print('DL01_DIR:', DL01_DIR)
print('DL02_DIR:', DL02_DIR)
print('NPZ exists:', NPZ_FILE.exists())
print('Patch metadata exists:', PATCH_METADATA_FILE.exists())
print('Label class raster exists:', LABEL_CLASS_TIF.exists())


## Step 4 — Load DL-01 patch dataset

The loader below is robust to several possible key names inside the `.npz` file. It will automatically detect:

- input patches `X`
- classification labels `y_class`
- optional regression labels `y_score`


In [ ]:
# ============================================================
# Step 4 — Load patch dataset and metadata
# ============================================================

if not NPZ_FILE.exists():
    raise FileNotFoundError(f'Patch dataset not found: {NPZ_FILE}')

npz = np.load(NPZ_FILE)
print('NPZ keys:', npz.files)

def find_key(possible_keys, available_keys):
    for k in possible_keys:
        if k in available_keys:
            return k
    return None

x_key = find_key(['X', 'X_patches', 'input_patches', 'images', 'x'], npz.files)
y_class_key = find_key(['y_class', 'y_class_patches', 'label_class_patches', 'masks', 'y_seg', 'labels_class'], npz.files)
y_score_key = find_key(['y_score', 'y_score_patches', 'label_score_patches', 'labels_score', 'scores'], npz.files)

if x_key is None or y_class_key is None:
    raise KeyError(f'Could not detect X or y_class keys. Available keys: {npz.files}')

X = npz[x_key]
y_class = npz[y_class_key]
y_score = npz[y_score_key] if y_score_key is not None else None

print('Detected X key:', x_key)
print('Detected y_class key:', y_class_key)
print('Detected y_score key:', y_score_key)
print('X shape:', X.shape, X.dtype)
print('y_class shape:', y_class.shape, y_class.dtype)
if y_score is not None:
    print('y_score shape:', y_score.shape, y_score.dtype)

# Ensure X is N, C, H, W
if X.ndim != 4:
    raise ValueError(f'Expected X to be 4D, got shape {X.shape}')

# Common case from DL-01 may be N,H,W,C. Convert to N,C,H,W.
if X.shape[-1] <= 64 and X.shape[1] != X.shape[-1]:
    print('Converting X from N,H,W,C to N,C,H,W')
    X = np.transpose(X, (0, 3, 1, 2))

# Ensure y_class is N,H,W
if y_class.ndim == 4 and y_class.shape[1] == 1:
    y_class = y_class[:, 0, :, :]
elif y_class.ndim == 4 and y_class.shape[-1] == 1:
    y_class = y_class[:, :, :, 0]

print('Final X shape:', X.shape)
print('Final y_class shape:', y_class.shape)

N, C, H, W = X.shape
print(f'Number of patches: {N}, input bands: {C}, patch size: {H}x{W}')

metadata = pd.read_csv(PATCH_METADATA_FILE) if PATCH_METADATA_FILE.exists() else pd.DataFrame({'patch_id': [f'patch_{i:05d}' for i in range(N)]})
print('Metadata shape:', metadata.shape)
display(metadata.head())


In [ ]:
# ============================================================
# Step 5 — Patch dataset audit
# ============================================================

# Pixel-level class distribution excluding nodata
valid_pixels = y_class[y_class != CLASS_NODATA]
unique, counts = np.unique(valid_pixels, return_counts=True)
pixel_class_dist = pd.DataFrame({
    'class_id': unique.astype(int),
    'class_name': [CLASS_NAMES[int(i)] if int(i) < len(CLASS_NAMES) else f'class_{int(i)}' for i in unique],
    'pixel_count': counts.astype(int),
})
pixel_class_dist['pixel_fraction'] = pixel_class_dist['pixel_count'] / pixel_class_dist['pixel_count'].sum()

print('Pixel-level class distribution:')
display(pixel_class_dist)

if 'dominant_class_name' in metadata.columns:
    patch_class_dist = metadata['dominant_class_name'].value_counts().rename_axis('dominant_class_name').reset_index(name='patch_count')
else:
    dominant_class = []
    for mask in y_class:
        vals = mask[mask != CLASS_NODATA]
        if len(vals) == 0:
            dominant_class.append('NoData')
        else:
            dominant_class.append(CLASS_NAMES[int(np.bincount(vals.astype(int), minlength=NUM_CLASSES).argmax())])
    metadata['dominant_class_name'] = dominant_class
    patch_class_dist = metadata['dominant_class_name'].value_counts().rename_axis('dominant_class_name').reset_index(name='patch_count')

print('Patch-level dominant class distribution:')
display(patch_class_dist)

# Save audit
with pd.ExcelWriter(DL02_DIR / 'dl02_initial_dataset_audit.xlsx', engine='openpyxl') as writer:
    pixel_class_dist.to_excel(writer, sheet_name='Pixel_Class_Distribution', index=False)
    patch_class_dist.to_excel(writer, sheet_name='Patch_Class_Distribution', index=False)
    metadata.head(200).to_excel(writer, sheet_name='Patch_Metadata_Head', index=False)

print('Saved:', DL02_DIR / 'dl02_initial_dataset_audit.xlsx')


## Step 6 — Train, validation, and test split

The split is stratified by the dominant patch class so that each suitability class remains represented across train, validation, and test sets.

Default split:

- Training: 70%
- Validation: 15%
- Test: 15%


In [ ]:
# ============================================================
# Step 6 — Split patches into train, validation, and test sets
# ============================================================

indices = np.arange(N)
stratify_labels = metadata['dominant_class_name'].values if 'dominant_class_name' in metadata.columns else None

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=stratify_labels
)

temp_strat = stratify_labels[temp_idx] if stratify_labels is not None else None
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_strat
)

split_df = pd.DataFrame({
    'patch_index': np.concatenate([train_idx, val_idx, test_idx]),
    'split': ['train'] * len(train_idx) + ['val'] * len(val_idx) + ['test'] * len(test_idx)
}).sort_values('patch_index')

metadata_with_split = metadata.copy()
metadata_with_split['patch_index'] = np.arange(N)
metadata_with_split = metadata_with_split.merge(split_df, on='patch_index', how='left')

print('Split counts:')
display(metadata_with_split['split'].value_counts().reset_index(name='patch_count'))

if 'dominant_class_name' in metadata_with_split.columns:
    display(pd.crosstab(metadata_with_split['dominant_class_name'], metadata_with_split['split']))

metadata_with_split.to_csv(DL02_DIR / 'patch_metadata_with_split.csv', index=False)
print('Saved:', DL02_DIR / 'patch_metadata_with_split.csv')


In [ ]:
# ============================================================
# Step 7 — PyTorch Dataset and DataLoader
# ============================================================

class PatchSegmentationDataset(Dataset):
    def __init__(self, X, y, indices, augment=False):
        self.X = X
        self.y = y
        self.indices = np.array(indices)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x = self.X[i].astype(np.float32)
        y = self.y[i].astype(np.int64)

        # Replace NaN/Inf in input with zero after normalization
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        # Simple spatial augmentation for training
        if self.augment:
            if random.random() < 0.5:
                x = np.flip(x, axis=2).copy()
                y = np.flip(y, axis=1).copy()
            if random.random() < 0.5:
                x = np.flip(x, axis=1).copy()
                y = np.flip(y, axis=0).copy()
            if random.random() < 0.5:
                k = random.choice([1, 2, 3])
                x = np.rot90(x, k=k, axes=(1, 2)).copy()
                y = np.rot90(y, k=k, axes=(0, 1)).copy()

        return torch.from_numpy(x), torch.from_numpy(y)

train_ds = PatchSegmentationDataset(X, y_class, train_idx, augment=True)
val_ds = PatchSegmentationDataset(X, y_class, val_idx, augment=False)
test_ds = PatchSegmentationDataset(X, y_class, test_idx, augment=False)
all_ds = PatchSegmentationDataset(X, y_class, np.arange(N), augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
all_loader = DataLoader(all_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

xb, yb = next(iter(train_loader))
print('Batch X:', xb.shape, xb.dtype)
print('Batch y:', yb.shape, yb.dtype)


In [ ]:
# ============================================================
# Step 8 — Class weights for imbalanced pixels
# ============================================================

pixel_counts = np.zeros(NUM_CLASSES, dtype=np.float64)
for c in range(NUM_CLASSES):
    pixel_counts[c] = np.sum(valid_pixels == c)

# Inverse-frequency weights, normalized
weights = pixel_counts.sum() / (NUM_CLASSES * np.maximum(pixel_counts, 1))
weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

class_weight_df = pd.DataFrame({
    'class_id': np.arange(NUM_CLASSES),
    'class_name': CLASS_NAMES,
    'pixel_count': pixel_counts.astype(int),
    'class_weight': weights,
})

display(class_weight_df)
class_weight_df.to_csv(DL02_DIR / 'class_weights.csv', index=False)


## Step 9 — Define models

Two models are used:

1. **U-Net baseline**: a compact U-Net implemented directly in PyTorch.
2. **DeepLabV3+**: implemented using `segmentation_models_pytorch` with `encoder_weights=None`, so it does not require ImageNet weights and supports 21-channel geospatial input.

If `segmentation_models_pytorch` fails to load, the notebook will skip DeepLabV3+ and continue with U-Net.


In [ ]:
# ============================================================
# Step 9 — U-Net model definition
# ============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, in_channels, num_classes, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base*2)
        self.enc3 = DoubleConv(base*2, base*4)
        self.enc4 = DoubleConv(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base*8, base*16)
        self.up4 = nn.ConvTranspose2d(base*16, base*8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base*16, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out(d1)


In [ ]:
# ============================================================
# Step 10 — DeepLabV3+ model definition using segmentation_models_pytorch
# ============================================================

try:
    import segmentation_models_pytorch as smp
    SMP_AVAILABLE = True
    print('segmentation_models_pytorch is available.')
except Exception as e:
    SMP_AVAILABLE = False
    print('segmentation_models_pytorch is not available:', e)

def build_unet():
    return UNetSmall(in_channels=C, num_classes=NUM_CLASSES, base=32)

def build_deeplabv3plus():
    if not SMP_AVAILABLE:
        return None
    return smp.DeepLabV3Plus(
        encoder_name='resnet18',
        encoder_weights=None,
        in_channels=C,
        classes=NUM_CLASSES,
        activation=None,
    )


In [ ]:
# ============================================================
# Step 11 — Metrics and training utilities
# ============================================================

def update_confusion_matrix(cm, y_true, y_pred, num_classes=NUM_CLASSES, ignore_index=CLASS_NODATA):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    mask = y_true != ignore_index
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    valid = (y_true >= 0) & (y_true < num_classes)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) == 0:
        return cm
    cm += confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
    return cm

def metrics_from_cm(cm):
    cm = cm.astype(np.float64)
    total = cm.sum()
    acc = np.trace(cm) / total if total > 0 else np.nan
    precision = np.diag(cm) / np.maximum(cm.sum(axis=0), 1)
    recall = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    iou = np.diag(cm) / np.maximum(cm.sum(axis=1) + cm.sum(axis=0) - np.diag(cm), 1)
    return {
        'accuracy': acc,
        'macro_precision': np.nanmean(precision),
        'macro_recall': np.nanmean(recall),
        'macro_f1': np.nanmean(f1),
        'mean_iou': np.nanmean(iou),
        'class_precision': precision,
        'class_recall': recall,
        'class_f1': f1,
        'class_iou': iou,
    }

@torch.no_grad()
def evaluate_model(model, loader, criterion=None):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        if criterion is not None:
            loss = criterion(logits, y)
            total_loss += loss.item()
            n_batches += 1
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
        y_np = y.detach().cpu().numpy()
        cm = update_confusion_matrix(cm, y_np, pred)

    metrics = metrics_from_cm(cm)
    metrics['loss'] = total_loss / max(n_batches, 1) if criterion is not None else np.nan
    metrics['confusion_matrix'] = cm
    return metrics

def train_model(model, model_name, train_loader, val_loader, epochs):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=CLASS_NODATA)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

    best_val_miou = -np.inf
    best_path = DL02_DIR / f'{model_name}_best.pth'
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f'{model_name} epoch {epoch}/{epochs}', leave=False)
        for x, y in pbar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({'loss': train_loss / n_batches})

        train_loss = train_loss / max(n_batches, 1)
        val_metrics = evaluate_model(model, val_loader, criterion)
        scheduler.step(val_metrics['mean_iou'])

        row = {
            'model': model_name,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
            'val_mean_iou': val_metrics['mean_iou'],
            'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        print(f"{model_name} epoch {epoch:02d}: train_loss={train_loss:.4f}, val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['accuracy']:.4f}, val_macroF1={val_metrics['macro_f1']:.4f}, val_mIoU={val_metrics['mean_iou']:.4f}")

        if val_metrics['mean_iou'] > best_val_miou:
            best_val_miou = val_metrics['mean_iou']
            torch.save({
                'model_name': model_name,
                'model_state_dict': model.state_dict(),
                'in_channels': C,
                'num_classes': NUM_CLASSES,
                'class_names': CLASS_NAMES,
                'best_val_mean_iou': best_val_miou,
                'config': {
                    'batch_size': BATCH_SIZE,
                    'learning_rate': LEARNING_RATE,
                    'weight_decay': WEIGHT_DECAY,
                    'patch_size': PATCH_SIZE,
                    'seed': SEED,
                }
            }, best_path)
            print(f'  Saved best model: {best_path}')

    history_df = pd.DataFrame(history)
    history_df.to_csv(DL02_DIR / f'{model_name}_training_history.csv', index=False)
    return model, history_df, best_path


In [ ]:
# ============================================================
# Step 12 — Train U-Net baseline
# ============================================================

trained_models = {}
histories = []
best_paths = {}

if TRAIN_UNET:
    unet = build_unet()
    print(unet.__class__.__name__)
    unet, hist_unet, unet_best_path = train_model(unet, 'UNet', train_loader, val_loader, EPOCHS_UNET)
    trained_models['UNet'] = unet
    histories.append(hist_unet)
    best_paths['UNet'] = unet_best_path
else:
    print('Skipping U-Net training.')


In [ ]:
# ============================================================
# Step 13 — Train DeepLabV3+ model
# ============================================================

if TRAIN_DEEPLABV3PLUS and SMP_AVAILABLE:
    deeplab = build_deeplabv3plus()
    print(deeplab.__class__.__name__)
    deeplab, hist_deeplab, deeplab_best_path = train_model(deeplab, 'DeepLabV3Plus', train_loader, val_loader, EPOCHS_DEEPLAB)
    trained_models['DeepLabV3Plus'] = deeplab
    histories.append(hist_deeplab)
    best_paths['DeepLabV3Plus'] = deeplab_best_path
else:
    print('Skipping DeepLabV3+ training. SMP_AVAILABLE:', SMP_AVAILABLE)


In [ ]:
# ============================================================
# Step 14 — Load best checkpoints and evaluate on test set
# ============================================================

# Rebuild models and load best states for fair test evaluation
best_models = {}
for model_name, ckpt_path in best_paths.items():
    if not Path(ckpt_path).exists():
        continue
    if model_name == 'UNet':
        model = build_unet()
    elif model_name == 'DeepLabV3Plus':
        model = build_deeplabv3plus()
    else:
        continue
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    best_models[model_name] = model

criterion_eval = nn.CrossEntropyLoss(weight=class_weights, ignore_index=CLASS_NODATA)

summary_rows = []
class_metric_rows = []
conf_mats = {}

for model_name, model in best_models.items():
    val_metrics = evaluate_model(model, val_loader, criterion_eval)
    test_metrics = evaluate_model(model, test_loader, criterion_eval)
    conf_mats[model_name] = test_metrics['confusion_matrix']

    summary_rows.append({
        'model': model_name,
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
        'val_macro_f1': val_metrics['macro_f1'],
        'val_mean_iou': val_metrics['mean_iou'],
        'test_loss': test_metrics['loss'],
        'test_accuracy': test_metrics['accuracy'],
        'test_macro_precision': test_metrics['macro_precision'],
        'test_macro_recall': test_metrics['macro_recall'],
        'test_macro_f1': test_metrics['macro_f1'],
        'test_mean_iou': test_metrics['mean_iou'],
    })

    for i, cls in enumerate(CLASS_NAMES):
        class_metric_rows.append({
            'model': model_name,
            'class_id': i,
            'class_name': cls,
            'precision': test_metrics['class_precision'][i],
            'recall': test_metrics['class_recall'][i],
            'f1': test_metrics['class_f1'][i],
            'iou': test_metrics['class_iou'][i],
        })

performance_df = pd.DataFrame(summary_rows)
class_metrics_df = pd.DataFrame(class_metric_rows)

print('Model performance summary:')
display(performance_df)
print('Class-level metrics:')
display(class_metrics_df)

performance_df.to_csv(DL02_DIR / 'dl02_model_performance_summary.csv', index=False)
class_metrics_df.to_csv(DL02_DIR / 'dl02_class_level_metrics.csv', index=False)


In [ ]:
# ============================================================
# Step 15 — Plot training curves and confusion matrices
# ============================================================

if len(histories) > 0:
    all_history = pd.concat(histories, ignore_index=True)
    all_history.to_csv(DL02_DIR / 'dl02_training_history_all_models.csv', index=False)

    for metric in ['train_loss', 'val_loss', 'val_accuracy', 'val_macro_f1', 'val_mean_iou']:
        plt.figure(figsize=(8, 5))
        for model_name in all_history['model'].unique():
            sub = all_history[all_history['model'] == model_name]
            plt.plot(sub['epoch'], sub[metric], marker='o', label=model_name)
        plt.title(metric.replace('_', ' ').title())
        plt.xlabel('Epoch')
        plt.ylabel(metric)
        plt.grid(True, alpha=0.3)
        plt.legend()
        out = DL02_DIR / f'training_curve_{metric}.png'
        plt.tight_layout()
        plt.savefig(out, dpi=300)
        plt.show()
        print('Saved:', out)

# Confusion matrix plots
for model_name, cm in conf_mats.items():
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation='nearest')
    plt.title(f'Confusion Matrix: {model_name}')
    plt.colorbar()
    tick_marks = np.arange(NUM_CLASSES)
    plt.xticks(tick_marks, CLASS_NAMES, rotation=45, ha='right')
    plt.yticks(tick_marks, CLASS_NAMES)
    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > thresh else 'black')
    plt.ylabel('Observed')
    plt.xlabel('Predicted')
    plt.tight_layout()
    out = DL02_DIR / f'confusion_matrix_{model_name}.png'
    plt.savefig(out, dpi=300)
    plt.show()
    print('Saved:', out)


In [ ]:
# ============================================================
# Step 16 — Predict all selected patches and reconstruct prediction raster
# ============================================================

@torch.no_grad()
def predict_all_patches(model, loader):
    model.eval()
    preds = []
    probs_max = []
    for x, y in tqdm(loader, desc='Predicting all patches'):
        x = x.to(device, non_blocking=True)
        logits = model(x)
        prob = torch.softmax(logits, dim=1)
        pred = torch.argmax(prob, dim=1)
        conf = torch.max(prob, dim=1).values
        preds.append(pred.cpu().numpy().astype(np.uint8))
        probs_max.append(conf.cpu().numpy().astype(np.float32))
    return np.concatenate(preds, axis=0), np.concatenate(probs_max, axis=0)

# Select best model by test mean IoU
if len(performance_df) > 0:
    best_model_name = performance_df.sort_values('test_mean_iou', ascending=False).iloc[0]['model']
    print('Best model by test_mean_iou:', best_model_name)
    best_model = best_models[best_model_name]

    all_pred, all_conf = predict_all_patches(best_model, all_loader)
    np.save(DL02_DIR / f'{best_model_name}_all_patch_predictions.npy', all_pred)
    np.save(DL02_DIR / f'{best_model_name}_all_patch_confidence.npy', all_conf)

    print('All patch predictions:', all_pred.shape)

    # Reconstruct raster from non-overlapping patches using patch metadata row/col.
    if LABEL_CLASS_TIF.exists() and {'row', 'col'}.issubset(metadata.columns):
        with rasterio.open(LABEL_CLASS_TIF) as src:
            profile = src.profile.copy()
            out_shape = (src.height, src.width)

        pred_raster = np.full(out_shape, CLASS_NODATA, dtype=np.uint8)
        conf_raster = np.full(out_shape, np.nan, dtype=np.float32)

        for i, row in metadata.iterrows():
            r = int(row['row'])
            c0 = int(row['col'])
            patch_pred = all_pred[i]
            patch_conf = all_conf[i]
            rr = min(r + patch_pred.shape[0], out_shape[0])
            cc = min(c0 + patch_pred.shape[1], out_shape[1])
            h2 = rr - r
            w2 = cc - c0
            pred_raster[r:rr, c0:cc] = patch_pred[:h2, :w2]
            conf_raster[r:rr, c0:cc] = patch_conf[:h2, :w2]

        profile.update(count=1, dtype='uint8', nodata=CLASS_NODATA, compress='lzw')
        pred_tif = DL02_DIR / f'{best_model_name}_predicted_potential_class.tif'
        with rasterio.open(pred_tif, 'w', **profile) as dst:
            dst.write(pred_raster, 1)
        print('Saved prediction raster:', pred_tif)

        conf_profile = profile.copy()
        conf_profile.update(dtype='float32', nodata=np.nan, compress='lzw')
        conf_tif = DL02_DIR / f'{best_model_name}_prediction_confidence.tif'
        with rasterio.open(conf_tif, 'w', **conf_profile) as dst:
            dst.write(conf_raster.astype(np.float32), 1)
        print('Saved confidence raster:', conf_tif)
else:
    print('No trained model available for prediction raster reconstruction.')


In [ ]:
# ============================================================
# Step 17 — Save final DL-02 summary workbook
# ============================================================

output_files = []
for p in sorted(DL02_DIR.glob('*')):
    if p.is_file():
        output_files.append({'file_name': p.name, 'path': str(p)})
output_files_df = pd.DataFrame(output_files)

summary_items = {
    'input_npz': str(NPZ_FILE),
    'number_of_patches': int(N),
    'input_band_count': int(C),
    'patch_height': int(H),
    'patch_width': int(W),
    'num_classes': int(NUM_CLASSES),
    'class_names': ', '.join(CLASS_NAMES),
    'train_patches': int(len(train_idx)),
    'val_patches': int(len(val_idx)),
    'test_patches': int(len(test_idx)),
    'batch_size': int(BATCH_SIZE),
    'epochs_unet': int(EPOCHS_UNET),
    'epochs_deeplabv3plus': int(EPOCHS_DEEPLAB),
    'learning_rate': float(LEARNING_RATE),
    'best_model_by_test_mean_iou': best_model_name if 'best_model_name' in globals() else None,
}
summary_df = pd.DataFrame(list(summary_items.items()), columns=['item', 'value'])

xlsx_out = DL02_DIR / 'dl02_training_summary.xlsx'
with pd.ExcelWriter(xlsx_out, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='DL02 Summary', index=False)
    pixel_class_dist.to_excel(writer, sheet_name='Pixel Class Dist', index=False)
    patch_class_dist.to_excel(writer, sheet_name='Patch Class Dist', index=False)
    class_weight_df.to_excel(writer, sheet_name='Class Weights', index=False)
    metadata_with_split.to_excel(writer, sheet_name='Patch Split', index=False)
    if 'performance_df' in globals():
        performance_df.to_excel(writer, sheet_name='Model Performance', index=False)
    if 'class_metrics_df' in globals():
        class_metrics_df.to_excel(writer, sheet_name='Class Metrics', index=False)
    output_files_df.to_excel(writer, sheet_name='Output Files', index=False)

print('Saved:', xlsx_out)
display(summary_df)
display(performance_df if 'performance_df' in globals() else pd.DataFrame())
